<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/rnn/wip_rnn_variable_length_bit_flip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Variable Length Bit Flipper

In this notebook we will train a RNN to learn how to flip bits in sequences of arbritary length.

## Setup

In [2]:
!pip install wandb tsilva-notebook-utils==0.0.13 > /dev/null

Load secrets:

In [3]:
from tsilva_notebook_utils.colab import load_secrets_into_env
load_secrets_into_env([
    'HF_TOKEN',
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define configuration:

In [4]:
import os
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    # @markdown ### 🌱 Reproducibility Settings

    # @markdown Random seed for reproducibility
    seed = 42  # @param {type:"integer"}

    # @markdown ### 🧩 Dataset Settings

    # @markdown Total size of the synthetic dataset
    dataset_size = 10_000  # @param {type:"integer"}

    # @markdown Minimum length of input sequence (time steps)
    min_seq_length = 2  # @param {type:"integer"}

    # @markdown Maximum length of input sequence (time steps)
    max_seq_length = 100  # @param {type:"integer"}

    # @markdown ### 🏋️ Training Settings

    # @markdown Number of training epochs
    n_epochs = 5  # @param {type:"integer"}

    # @markdown Batch size for training
    batch_size = 256  # @param {type:"integer"}

    # @markdown Learning rate for the optimizer
    learning_rate = 0.001  # @param {type:"number"}

    # @markdown ### 🧠 Model Architecture Settings

    # @markdown Input size (number of input features)
    input_size = 1  # @param {type:"integer"}

    # @markdown Hidden layer size
    hidden_size = 16  # @param {type:"integer"}

    # @markdown Output size (number of output features)
    output_size = 1  # @param {type:"integer"}

    # Generate notebook id from notebook title
    os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'dataset_size': dataset_size,
        'min_seq_length': min_seq_length,
        'max_seq_length': max_seq_length,
        'learning_rate': learning_rate,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size
    }

CONFIG = setup_config()

First let's build a dataset:

In [5]:
import random
import torch
from torch.utils.data import Dataset, DataLoader

class BitFlipDataset(Dataset):
    def __init__(
        self,
        dataset_size=None,
        min_seq_length=None,
        max_seq_length=None
    ):
        # Use default values from CONFIG if arguments are not provided
        if dataset_size is None: dataset_size = CONFIG['dataset_size']
        if min_seq_length is None: min_seq_length = CONFIG['min_seq_length']
        if max_seq_length is None: max_seq_length = CONFIG['max_seq_length']

        self.data = []  # Initialize an empty list to store data samples

        # Generate the dataset
        for _ in range(dataset_size):
            # Randomly select a sequence length within the given range
            seq_len = random.randint(min_seq_length, max_seq_length)

            # Generate a random binary sequence (0s and 1s), shape: (seq_len, 1)
            X = torch.randint(0, 2, (seq_len, 1)).float()

            # Create the target sequence by flipping the bits (1 -> 0, 0 -> 1)
            Y = 1.0 - X

            # Append the input-output pair to the dataset
            self.data.append((X, Y))

    # Return the total number of samples in the dataset
    def __len__(self):
        return len(self.data)

    # Retrieve a sample by index
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = BitFlipDataset()
train_dataset[0]

(tensor([[0.],
         [0.],
         [1.],
         [1.],
         [0.],
         [1.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [1.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [1.],
         [0.],
         [0.],
         [1.],
         [1.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [1.],
         [0.],
         [0.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [0.],
         [0.],
         [1.],
         [0.],
         [0.],
         [0.],
         [1.]]),
 tensor([[1.],
         [1.],
         [0.],
         [0.],
         [1.],
         [0.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
        

Now let's build a data loader for the dataset. To use a data loader on samples of different length we'll need to provide a custom collator. By default pytorch will try to stack the tensors when loading a batch, but you can't stack vectors of different length, therefore we need to pad them to max batch length:

In [6]:
import torch.nn as nn

def collate_fn(batch):
    seqs_x, seqs_y = zip(*batch)
    padded_x = nn.utils.rnn.pad_sequence(seqs_x, batch_first=True, padding_value=-1)
    padded_y = nn.utils.rnn.pad_sequence(seqs_y, batch_first=True, padding_value=-1)
    return padded_x, padded_y

collate_fn([
    (torch.tensor([2, 2, 2, 2], dtype=torch.long), torch.tensor([3, 3, 3, 3], dtype=torch.long)),
    (torch.tensor([1, 1], dtype=torch.long), torch.tensor([2, 2], dtype=torch.long)),
    (torch.tensor([3, 3, 3], dtype=torch.long), torch.tensor([4, 4, 4], dtype=torch.long))
])

(tensor([[ 2,  2,  2,  2],
         [ 1,  1, -1, -1],
         [ 3,  3,  3, -1]]),
 tensor([[ 3,  3,  3,  3],
         [ 2,  2, -1, -1],
         [ 4,  4,  4, -1]]))

Let's already address how we deal with padding when calculating the loss function. We don't want these -1s to count towards the loss, we just want to ignore them. To do so we just create a mask tensor that has 1s for non padding value positions and 0s for padding positions. Let's try:

In [7]:
t1 = torch.tensor([[1, 2, -1, -1], [-1, 4, 3, -1]])
mask = t1 != -1
t1 * mask

tensor([[1, 2, 0, 0],
        [0, 4, 3, 0]])

Test the data loader with collate function:

In [8]:
batch_size = 10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
x, y = next(iter(train_loader))
x.shape, x[2]

(torch.Size([10, 95, 1]),
 tensor([[ 0.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 1.],
         [ 0.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 0.],
         [ 0.],
         [ 1.],
         [ 1.]

Let's create the model, the model will be a RNN module which processes the sequence and outputs an hidden state, and linear module, which will convert that hidden state into the output size (in our case a single value, which want to be either 0 or 1):

In [9]:
# Display configuration
for key in ['input_size', 'hidden_size', 'output_size']:
    print(f"{key.replace('_', ' ').title()}: {CONFIG[key]}")

# Prepare input tensor
x = torch.tensor([[[1], [2]]]).float()
print(f"Input shape: {x.shape}")

# Initialize RNN
rnn = nn.RNN(CONFIG['input_size'], CONFIG['hidden_size'], batch_first=True)

# Forward pass
hidden_states, last_hidden_state = rnn(x)

# Display output shapes and values
print(f"Hidden states shape: {hidden_states.shape}")
print(f"Last hidden state shape: {last_hidden_state.shape}")
print("Hidden states:\n", hidden_states)
print("Last hidden state:\n", last_hidden_state)

linear = nn.Linear(CONFIG['hidden_size'], CONFIG['output_size'])
output = linear(last_hidden_state)
output.shape, output

Input Size: 1
Hidden Size: 16
Output Size: 1
Input shape: torch.Size([1, 2, 1])
Hidden states shape: torch.Size([1, 2, 16])
Last hidden state shape: torch.Size([1, 1, 16])
Hidden states:
 tensor([[[-0.1094, -0.2167,  0.0548, -0.2507, -0.3957, -0.2378,  0.4219,
          -0.2313, -0.1378,  0.2925, -0.2820, -0.0747,  0.0312,  0.1259,
          -0.1131, -0.3629],
         [ 0.1360, -0.2231, -0.1646, -0.5552, -0.5034, -0.1800,  0.4572,
          -0.3196,  0.1136,  0.3702, -0.3836, -0.2318,  0.4163, -0.0599,
          -0.0486, -0.4187]]], grad_fn=<TransposeBackward1>)
Last hidden state:
 tensor([[[ 0.1360, -0.2231, -0.1646, -0.5552, -0.5034, -0.1800,  0.4572,
          -0.3196,  0.1136,  0.3702, -0.3836, -0.2318,  0.4163, -0.0599,
          -0.0486, -0.4187]]], grad_fn=<StackBackward0>)


(torch.Size([1, 1, 1]), tensor([[[-0.2576]]], grad_fn=<ViewBackward0>))

Create the model:

In [10]:
class BitFlipperRNN(nn.Module):
    def __init__(
        self,
        input_size=None,
        hidden_size=None,
        output_size=None
    ):
        super().__init__()

        # Use default values from CONFIG if not provided
        if input_size is None: input_size = CONFIG['input_size']
        if hidden_size is None: hidden_size = CONFIG['hidden_size']
        if output_size is None: output_size = CONFIG['output_size']

        # Define an RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        # Fully connected layer to map hidden state output to desired output size
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Pass the input through the RNN layer
        hidden_states, _ = self.rnn(x)  # out: (batch, seq_len, hidden_size)

        # Pass the RNN output through the fully connected layer
        output = self.fc(hidden_states)  # output: (batch, seq_len, output_size)

        return output, hidden_states

model = BitFlipperRNN()

Login to wandb and show dashboard before we start training:

In [11]:
from tsilva_notebook_utils.wandb import init_with_defaults
init_with_defaults(CONFIG)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Train the model:

In [32]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import wandb

# Set model in training mode
model.train()

# Optionally: Watch the model to log gradients and model topology
#wandb.watch(model, log="all")

# Define the loss function as Mean Squared Error loss
loss_fn = nn.MSELoss()

# Set learning rate from configuration
learning_rate = CONFIG['learning_rate']

# Initialize the Adam optimizer with model parameters and the learning rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Get the number of training epochs from configuration
n_epochs = CONFIG['n_epochs']

# Training loop with progress bar using tqdm
with tqdm(range(n_epochs), desc="Training") as pbar:
    for epoch in pbar:
        losses = []
        accuracies = []

        # Iterate over batches of data from the training data loader
        for x_batch, y_batch in train_loader:
            # Forward pass: compute model predictions and capture hidden states
            outputs, hidden_states = model(x_batch)

            # Create a mask to ignore padded values (assuming -1 is used for padding)
            mask = (x_batch != -1).float() # TODO: move padding value to config

            # Compute the loss, applying the mask to ignore padded elements
            loss = loss_fn(outputs * mask, y_batch * mask)

            # Backpropagation step
            optimizer.zero_grad()  # Clear previous gradients
            loss.backward()        # Compute gradients
            optimizer.step()       # Update model parameters

            with torch.no_grad():
                cropped_x_batch = x_batch[mask.bool()]
                cropped_outputs = torch.round(torch.abs(outputs[mask.bool()]))
                accuracy = torch.numel(cropped_x_batch == cropped_outputs) / torch.numel(cropped_x_batch)
                accuracies.append(accuracy)

                # Accumulate the loss value for this batch
                losses.append(loss.item())

        # Compute average loss for the epoch
        avg_loss = sum(losses) / len(train_loader)
        avg_accuracy = sum(accuracies) / len(accuracies)

        # Log average loss to wandb
        #wandb.log({'epoch': epoch + 1, 'loss': avg_loss})

        # Update the progress bar with the current epoch and average loss
        pbar.set_postfix({'Epoch': epoch + 1, 'Loss': f'{avg_loss:.6f}', 'Accuracy': f'{avg_accuracy * 100:.2f}%'})

        if avg_accuracy == 1.0:
           print("\nTraining finished, model memorized dataset.")
           break

#wandb.finish()

Training:   0%|          | 0/5 [00:14<?, ?it/s, Epoch=1, Loss=0.000001, Accuracy=100.00%]

Training finished, model memorized dataset.


In [42]:
' '.join([1, 2])

TypeError: sequence item 0: expected str instance, int found

In [ ]:
while True:
    sequence_s = input()
    x = torch.tensor(list(map(int, sequence_s))).unsqueeze(-1).float()
    outputs, _ = model(x)
    prediction = torch.round(torch.abs(outputs))
    prediction = "".join([str(x) for x in prediction.int().squeeze().tolist()])
    print(prediction)

12
01
10101010101
01010101010
100000000000000000000000000000000001
011111111111111111111111111111111110
1000100101010101010110
0111011010101010101001
111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110111111111111111111111111
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001000000000000000000000000


Test the sequences:

In [17]:
from tsilva_notebook_utils.colab import notify_and_disconnect_after_timeout
notify_and_disconnect_after_timeout()

Starting idle timeout check. Will disconnect after 300 seconds of no interruption...


Idle Timeout:  30%|███       | 91/300 [01:31<03:29,  1.00s/s]


KeyboardInterrupt: 